In [1]:
import Pkg
Pkg.add(["JuMP", "GLPK", "Format"])

using JuMP
using GLPK
using Format

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed CodecBzip2 ───────── v0.8.5
   Installed GLPK_jll ─────────── v5.0.1+1
   Installed GLPK ─────────────── v1.2.1
   Installed MutableArithmetics ─ v1.8.0
   Installed MathOptInterface ─── v1.51.1
   Installed JuMP ─────────────── v1.30.1
  Installing 1 artifacts
   Installed artifact GLPK                     3.1 MiB
    Updating `~/.julia/environments/v1.12/Project.toml`
  [1fa38f19] + Format v1.3.7
  [60bf3e95] + GLPK v1.2.1
  [4076af6c] + JuMP v1.30.1
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [523fee87] + CodecBzip2 v0.8.5
  [60bf3e95] + GLPK v1.2.1
  [4076af6c] + JuMP v1.30.1
  [b8f27783] + MathOptInterface v1.51.1
  [d8a4904e] + MutableArithmetics v1.8.0
⌅ [e8aa6df9] + GLPK_jll v5.0.1+1
  [781609d7] + GMP_jll v6.3.0+2
        Info Packages marked with ⌅ have new versions available but compatibility constraints restrict them from upgrading. To see why use `status

In [5]:
function solve_min_distance_schedule(N, P; time_limit=nothing)

    # Constante BIG-M
    M = sum(P)

    model = Model(GLPK.Optimizer)

    # Limite de tempo opicional para entradas muito longas (em minutos)
    if time_limit !== nothing
        set_time_limit_sec(model, time_limit * 60.0)
    end

    #
    # Variáveis
    #

    @variable(model, D >= 0)

    # tempos de início
    @variable(model, S[1:N] >= 0)

    # variáveis binárias b(i,j), apenas para i<j
    @variable(model, b[1:N-1,2:N], Bin)

    #
    # Função objetivo
    #

    @objective(model, Min, D)

    #
    # Restrição (3)
    #

    @constraint(model, S[1] == 0)

    #
    # Restrição (1)
    #

    for i in 1:N
        @constraint(model, D >= S[i] + P[i])
    end

    #
    # Restrições (5) e (6)
    #

    for i in 1:N-1
        for j in i+1:N

            minP = min(P[i], P[j])

            # (5)
            @constraint(model,
                S[i] >= minP + S[j] - M*(1 - b[i,j])
            )

            # (6)
            @constraint(model,
                S[j] >= minP + S[i] - M*b[i,j]
            )
        end
    end

    optimize!(model)

    println()
    println("Status: ", termination_status(model))
    println()

    if termination_status(model) == MOI.OPTIMAL

        println("Makespan = ", value(D))
        println()

        println("Tempos de início")

        for i in 1:N
            println(format("Tarefa {:2d}: {:8.2f}", i, value(S[i])))
        end

        println()
        println("Valores de b(i,j)")

        for i in 1:N-1
            for j in i+1:N
                println("b($i,$j) = ", Int(round(value(b[i,j]))))
            end
        end

    else
        println("Nenhuma solução ótima encontrada.")
    end

end

#
# Exemplo
#

P = [91,
38,
58,
16,
5,
28,
35,
95,
18,
27,
38,
74,
73,
88,
55,
25,
8,
77,
39,
63,
35,
38,
80,
86,
77,
75,
74,
47,
42,
87,
57,
79,
92,
81,
96,
51,
4,
4,
18,
65,
86,
58,
7,
55,
99,
4,
62,
40,
81,
74]

solve_min_distance_schedule(50, P, time_limit=30.0)



Status: TIME_LIMIT

Nenhuma solução ótima encontrada.
